# Pattern 5: Memory-Enabled Agent (Thread Memory)

This notebook shows conversational memory using LangGraph checkpointers with Claude Haiku 4.5.
Memory is scoped by `thread_id`, so each thread keeps independent history.

```mermaid
flowchart LR
    I1[Turn 1] --> M[(Checkpoint Memory)]
    I2[Turn 2] --> M
    M --> A[Context-Aware Answer]
```

In [3]:
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import InMemorySaver

MODEL_NAME = "claude-haiku-4-5-20251001"
llm = ChatAnthropic(model=MODEL_NAME, temperature=0, max_tokens=1024)
checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=checkpointer,
)

In [2]:
thread_a = {"configurable": {"thread_id": "user-a"}}
thread_b = {"configurable": {"thread_id": "user-b"}}

agent.invoke(
    {"messages": [{"role": "user", "content": "My favorite language is Python."}]},
    config=thread_a,
)

agent.invoke(
    {"messages": [{"role": "user", "content": "My favorite language is Rust."}]},
    config=thread_b,
)

resp_a = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my favorite language?"}]},
    config=thread_a,
)
resp_b = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my favorite language?"}]},
    config=thread_b,
)

print("Thread A:", resp_a["messages"][-1].content)
print("Thread B:", resp_b["messages"][-1].content)

Thread A: Your favorite language is Python, as you mentioned in your first message.
Thread B: Your favorite language is Rust, as you mentioned at the start of our conversation.


## How the code cells map to the pattern
Cell 2 creates the Claude-backed agent and attaches the in-memory checkpointer that stores thread-scoped state.
Cell 3 uses two different `thread_id` values to prove that each conversation remembers its own history without leaking context across sessions.